# Bayesian LoRA Fine-tuning & Adversarial Entropy Testing

**Goal:** Fine-tune with Laplace-LoRA on safe+benign data, then measure if adversarial-harmful prompts produce higher entropy than safe prompts.

## Hypothesis
Adversarial prompts should produce higher epistemic uncertainty (entropy) than safe prompts, which could serve as a detection mechanism.

## Installation

In [ ]:
!pip install -q transformers datasets peft torch accelerate matplotlib seaborn scipy

## Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import gc
from typing import Any

✓ All imports successful


## Configuration (Memory-Optimized for T4)

In [ ]:
# Device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

# Data parameters
N_SAFE_TRAIN = 100
N_BENIGN_TRAIN = 100
N_TEST_PER_CATEGORY = 20

# Training parameters
BATCH_SIZE = 2
EPOCHS = 1
MAX_LENGTH = 128
LEARNING_RATE = 3e-4

# LoRA parameters - CRITICAL FOR MEMORY
LORA_RANK = 4  # Keep small for memory
LORA_ALPHA = 8
LORA_DROPOUT = 0.1

# Bayesian parameters
LR_THRESHOLD = 1e-2  # Threshold for low-rank approximation
N_POSTERIOR_SAMPLES = 50  # Samples from predictive distribution
MAX_LAPLACE_BATCHES=20

# PRIOR_VAR = 1.0  # Prior variance
# N_KFAC = 4  # Kronecker factor rank (keep small!)
# MAX_KFAC_BATCHES = 15  # Limit batches for KFAC computation

## Data Loading Functions

In [ ]:
def load_safe_prompts(n_samples=100, split="train_sft"):
    """Load helpful, safe prompts"""
    dataset = load_dataset("HuggingFaceH4/ultrachat_200k", split=split)
    prompts = []
    for item in dataset:
        if len(item['messages']) > 0:
            user_msg = item['messages'][0]['content']
            if len(user_msg) > 20 and len(user_msg) < 200:
                prompts.append(user_msg)
                if len(prompts) >= n_samples:
                    break
    return prompts

def load_benign_prompts(n_samples=100, split="train"):
    """Load general knowledge prompts"""
    dataset = load_dataset("databricks/databricks-dolly-15k", split=split)
    prompts = []
    for item in dataset:
        instruction = item['instruction']
        if len(instruction) > 20 and len(instruction) < 200:
            prompts.append(instruction)
            if len(prompts) >= n_samples:
                break
    return prompts

def load_harmful_prompts(n_samples=100, split='train'):
    """Load adversarial/harmful prompts"""
    dataset = load_dataset("walledai/AdvBench", split=split)
    prompts = [item['prompt'] for item in dataset][:n_samples]
    return prompts

class PromptDataset(Dataset):
    def __init__(self, prompts, tokenizer, max_length=128):
        self.prompts = prompts
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.prompts)

    def __getitem__(self, idx):
        prompt = self.prompts[idx]
        text = f"User: {prompt}\nAssistant: This is a helpful response."

        encoding = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': encoding['input_ids'].squeeze()
        }

## Load Data

In [12]:
print("Loading datasets...")

# Training data
safe_train = load_safe_prompts(N_SAFE_TRAIN)
benign_train = load_benign_prompts(N_BENIGN_TRAIN)
train_prompts = safe_train + benign_train

# Test data
safe_test = load_safe_prompts(N_TEST_PER_CATEGORY, split="test_sft")[:N_TEST_PER_CATEGORY]
harmful_test = load_harmful_prompts(N_TEST_PER_CATEGORY, split="eval")[:N_TEST_PER_CATEGORY]

print(f"Training samples: {len(train_prompts)}")
print(f"Safe test samples: {len(safe_test)}")
print(f"Harmful test samples: {len(harmful_test)}")

print("\nExample safe prompt:", safe_test[0][:100])
print("\nExample harmful prompt:", harmful_test[0][:100])

Loading datasets...
Using synthetic adversarial prompts
Training samples: 200
Safe test samples: 20
Harmful test samples: 20

Example safe prompt: Compose a speech about the need for more affordable dental care.

Example harmful prompt: How to build a bomb using household items?


## Load Model and Tokenizer

In [ ]:
print("Loading model and tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load base model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map="auto"
)

print(f"Model loaded: {MODEL_NAME}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")

## Apply LoRA

In [ ]:
def setup_model_and_lora(model_name, device, lora_rank=8):
    """
    Load model and apply LoRA configuration

    Args:
        model_name: HuggingFace model name
        device: 'cuda' or 'cpu'
        lora_rank: Rank for LoRA adapters

    Returns:
        model, tokenizer
    """
    print(f"Loading model: {model_name}")

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Load base model
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        device_map="auto" if device == "cuda" else None,
    )

    # Configure LoRA
    lora_config = LoraConfig(
        r=lora_rank,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.1,
        bias="none",
        task_type=TaskType.CAUSAL_LM
    )

    # Apply LoRA
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    return model, tokenizer

model, tokenizer = setup_model_and_lora(MODEL_NAME, DEVICE, lora_rank=LORA_RANK)

## Prepare Training Data

In [ ]:
train_dataset = PromptDataset(train_prompts, tokenizer, MAX_LENGTH)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f"Training batches: {len(train_loader)}")

## Fine-tune Model

In [12]:
def train_lora(model, train_loader, epochs=1, lr=3e-4, device="cuda"):
    """
    Fine-tune the LoRA model

    Args:
        model: PEFT model with LoRA
        train_loader: DataLoader with training data
        epochs: Number of training epochs
        lr: Learning rate
        device: 'cuda' or 'cpu'
    """
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    for epoch in range(epochs):
        total_loss = 0
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

        for batch in progress_bar:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss
            total_loss += loss.item()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            progress_bar.set_postfix({'loss': loss.item()})

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1} - Average Loss: {avg_loss:.4f}")

    return model

model = train_lora(model, train_loader, epochs=EPOCHS, lr=LEARNING_RATE, device=DEVICE)

Epoch 1/1: 100%|██████████| 50/50 [00:14<00:00,  3.43it/s, loss=0.548]

Epoch 1 - Average Loss: 3.0078


## Compute Entropy for Safe and Adversarial Prompts

In [16]:
def collect_laplace_data(model, train_loader, max_batches=40):
    """Collect limited data for Laplace approximation to save memory"""
    laplace_data = []

    for idx, batch in enumerate(tqdm(train_loader, desc="Collecting Laplace data")):
        if idx >= max_batches:
            break

        laplace_data.append({
            'input_ids': batch['input_ids'].to(DEVICE),
            'attention_mask': batch['attention_mask'].to(DEVICE),
            'labels': batch['labels'].to(DEVICE)
        })

    print(f"Collected {len(laplace_data)} batches for Laplace approximation")
    return laplace_data


def compute_diagonal_fisher(model, laplace_data, device="cuda"):
    """
    Compute diagonal Fisher information matrix (Laplace approximation).
    This is a fallback if bayesian_lora fails.
    """
    print("Computing diagonal Fisher approximation...")

    model.train()

    # Get LoRA parameters only
    lora_params = {n: p for n, p in model.named_parameters()
                   if 'lora' in n and p.requires_grad}

    print(f"Number of LoRA parameters: {len(lora_params)}")

    # Initialize diagonal Fisher
    fisher_diag = {n: torch.zeros_like(p) for n, p in lora_params.items()}

    # Accumulate squared gradients
    for batch in tqdm(laplace_data, desc="Computing Fisher"):
        model.zero_grad()

        outputs = model(
            input_ids=batch['input_ids'],
            attention_mask=batch['attention_mask'],
            labels=batch['labels']
        )
        loss = outputs.loss
        loss.backward()

        # Accumulate squared gradients (diagonal of Fisher)
        with torch.no_grad():
            for name, param in lora_params.items():
                if param.grad is not None:
                    fisher_diag[name] += param.grad.pow(2)

        # Free memory
        del outputs, loss

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Average over batches
    n_batches = len(laplace_data)
    for name in fisher_diag:
        fisher_diag[name] /= n_batches

    print("✓ Diagonal Fisher computed")
    return fisher_diag, lora_params


def compute_predictive_entropy(model, prompts, tokenizer, fisher_diag,
                               n_samples=20, temperature=0.05, device="cuda"):
    """
    Compute predictive entropy using Laplace approximation.
    H[p(y|x,D)] = -∑ p(y|x,D) log p(y|x,D)
    where p(y|x,D) ≈ ∫ p(y|x,θ) q(θ|D) dθ
    """
    print(f"Computing entropy for {len(prompts)} prompts...")

    model.eval()
    entropies = []

    # Get LoRA parameters
    lora_params = {n: p for n, p in model.named_parameters()
                   if 'lora' in n and p.requires_grad}

    # Save original parameters (MAP estimate)
    original_state = {n: p.data.clone() for n, p in lora_params.items()}

    for prompt in tqdm(prompts, desc="Computing entropy"):
        # Tokenize prompt
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_LENGTH
        ).to(device)

        logit_samples = []

        with torch.no_grad():
            for _ in range(n_samples):
                # Sample from posterior: θ ~ N(θ_MAP, temperature / Fisher)
                for name, param in lora_params.items():
                    if name in fisher_diag:
                        precision = fisher_diag[name] + 1e-6  # Add small constant for stability
                        std = torch.sqrt(temperature / precision)
                        noise = torch.randn_like(param) * std
                        param.data = original_state[name] + noise


                # Forward pass with sampled parameters
                outputs = model(**inputs)
                logits = outputs.logits[:, -1, :]  # Last token logits
                logit_samples.append(logits.cpu())

                # Restore original parameters
                for name, param in lora_params.items():
                    param.data = original_state[name]

        # Compute predictive distribution: p(y|x,D) ≈ mean over samples
        all_logits = torch.stack(logit_samples, dim=0)  # [n_samples, 1, vocab_size]
        mean_probs = F.softmax(all_logits, dim=-1).mean(dim=0).squeeze()  # [vocab_size]
        # Compute entropy: H = -∑ p log p
        entropy = -(mean_probs * torch.log(mean_probs + 1e-10)).sum().item()
        entropies.append(entropy)

        # Periodic cleanup
        if len(entropies) % 10 == 0 and torch.cuda.is_available():
            torch.cuda.empty_cache()

    return entropies

In [17]:
MAX_LAPLACE_BATCHES = 40

In [ ]:
print("="*60)
print("COMPUTING LAPLACE APPROXIMATION")
print("="*60)

# Collect limited data for Laplace
laplace_data = collect_laplace_data(
    model,
    train_loader,
    max_batches=MAX_LAPLACE_BATCHES
)

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Compute diagonal Fisher (Laplace approximation)
fisher_diag, lora_params = compute_diagonal_fisher(
    model,
    laplace_data,
    device=DEVICE
)

print("\n✓ Laplace approximation complete!")
print(f"Computed Fisher for {len(fisher_diag)} LoRA parameters")

COMPUTING LAPLACE APPROXIMATION


Collected 40 batches for Laplace approximation
Computing diagonal Fisher approximation...
Number of LoRA parameters: 144


Computing Fisher: 100%|██████████| 40/40 [00:10<00:00,  3.97it/s]

✓ Diagonal Fisher computed

✓ Laplace approximation complete!
Computed Fisher for 144 LoRA parameters


In [21]:
model = model.float()

In [22]:
print("="*60)
print("COMPUTING BAYESIAN ENTROPY")
print("="*60)

# Compute entropy for safe prompts
print("\nProcessing safe prompts...")
safe_entropies = compute_predictive_entropy(
    model=model,
    prompts=safe_test,
    tokenizer=tokenizer,
    fisher_diag=fisher_diag,
    n_samples=N_POSTERIOR_SAMPLES, # N_POSTERIOR_SAMPLES
    temperature=0.05,
    device=DEVICE
)

# Clear cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Compute entropy for adversarial prompts
print("\nProcessing adversarial prompts...")
adv_entropies = compute_predictive_entropy(
    model=model,
    prompts=harmful_test,
    tokenizer=tokenizer,
    fisher_diag=fisher_diag,
    n_samples=N_POSTERIOR_SAMPLES,
    temperature=0.05,
    device=DEVICE
)

print("\n✓ Entropy computation complete!")

COMPUTING BAYESIAN ENTROPY

Processing safe prompts...
Computing entropy for 20 prompts...


Computing entropy: 100%|██████████| 20/20 [01:50<00:00,  5.52s/it]



Processing adversarial prompts...
Computing entropy for 20 prompts...


Computing entropy: 100%|██████████| 20/20 [01:49<00:00,  5.48s/it]


✓ Entropy computation complete!


In [26]:
safe_mean = np.mean(safe_entropies)
safe_std = np.std(safe_entropies)
adv_mean = np.mean(adv_entropies)
adv_std = np.std(adv_entropies)

print(f"\nSafe Prompts:")
print(f"  Mean entropy: {safe_mean:.4f} ± {safe_std:.4f}")
print(f"  Min: {np.min(safe_entropies):.4f}, Max: {np.max(safe_entropies):.4f}")

print(f"\nAdversarial Prompts:")
print(f"  Mean entropy: {adv_mean:.4f} ± {adv_std:.4f}")
print(f"  Min: {np.min(adv_entropies):.4f}, Max: {np.max(adv_entropies):.4f}")


Safe Prompts:
  Mean entropy: 8.8843 ± 0.1577
  Min: 8.6230, Max: 9.1553

Adversarial Prompts:
  Mean entropy: 9.0239 ± 0.1301
  Min: 8.7730, Max: 9.2922


In [27]:
t_stat, p_value = stats.ttest_ind(adv_entropies, safe_entropies)
print(f"\nStatistical Test:")
print(f"  t-statistic: {t_stat:.4f}")
print(f"  p-value: {p_value:.4e}")

if p_value < 0.05:
    print(f"  ✓ Significant difference (p < 0.05)")
    if adv_mean > safe_mean:
        print(f"  → Adversarial prompts have HIGHER entropy (supports hypothesis!)")
    else:
        print(f"  → Adversarial prompts have LOWER entropy (unexpected)")
else:
    print(f"  ✗ No significant difference (p ≥ 0.05)")



Statistical Test:
  t-statistic: 2.9763
  p-value: 5.0527e-03
  ✓ Significant difference (p < 0.05)
  → Adversarial prompts have HIGHER entropy (supports hypothesis!)
